# CSC14120 – LẬP TRÌNH SONG SONG (PARALLEL PROGRAMMING)
## FINAL PROJECT: PANORAMA IMAGE STITCHING
### BÁO CÁO GIỮA KỲ: PROPOSAL VÀ PHIÊN BẢN TUẦN TỰ (V0 BASELINE)
**Trọng số:** 10% tổng điểm đồ án | **Hạn nộp:** 23:59 Cuối tuần thứ 7


---
## 0. SETUP CELL (MÔI TRƯỜNG THỰC THI & BIÊN DỊCH)
*Mục 4 yêu cầu: "One source of truth for code. The notebook must build the program from the submitted src/ files, not from separate copies written inside the notebook. Reproducible with Runtime -> Run all on a fresh Colab session."*

In [ ]:
# Setup environment and compile C++/CUDA source files
import os, sys

# Kiểm tra và sao chép thư mục src nếu cần thiết (hoặc mount từ Google Drive)
# from google.colab import drive
# drive.mount("/content/drive")
# !cp -r /content/drive/MyDrive/GroupXX/src /content/src

!mkdir -p outputs_v0
!echo "[INFO] Đang biên dịch mã nguồn V0 bằng nvcc..."
!cd src && nvcc -O3 -std=c++14 -w main.cu stb_impl.cpp v0_frontend.cpp v0_backend.cpp v0_stitching.cpp -o ../stitch -lm
!ls -lh stitch


---
## 3.1 THÔNG TIN NHÓM VÀ KẾ HOẠCH LÀM VIỆC (TEAM INFO & WORK PLAN)

### 1. Bảng thông tin thành viên:
| Vai trò | Họ tên | MSSV | Email |
| :--- | :--- | :--- | :--- |
| **Thành viên 1** (Back-end Lead) | `[Điền tên]` | `[Điền MSSV]` | `[Điền email]` |
| **Thành viên 2** (Front-end Lead) | `[Điền tên]` | `[Điền MSSV]` | `[Điền email]` |
| **Thành viên 3** (System & Data Lead) | `[Điền tên]` | `[Điền MSSV]` | `[Điền email]` |

### 2. Kế hoạch phân công công việc từ Giữa kỳ đến Cuối kỳ:
| Phiên bản / Mốc thời gian | Thành viên 1 (Back-end Lead) | Thành viên 2 (Front-end Lead) | Thành viên 3 (System & Data Lead) |
| :--- | :--- | :--- | :--- |
| **Midterm (Tuần 7)**<br>*V0 - Sequential Baseline* | Hiện thực V0 cho Stages 4, 5, 6; đề xuất kế hoạch GPU cho Matching, RANSAC, Warping. | Hiện thực V0 cho Stages 1, 2, 3; đề xuất kế hoạch GPU cho Preprocessing, Detection, Description. | Thu thập dataset 24 ảnh; dựng khung `main.cu`; đo thời gian V0; phân tích Amdahl; hoàn thiện Notebook. |
| **V1: Basic CUDA (Tuần 8 - 9)**<br>*Chuyển toàn bộ pipeline sang GPU* | Chuyển Stages 4, 5, 6 sang các CUDA kernel cơ bản (1 match/thread, 1 pixel/thread). | Chuyển Stages 1, 2, 3 sang các CUDA kernel cơ bản (1 pixel/thread, trích xuất descriptor trên GPU). | Đo thời gian bằng CUDA Events cho từng kernel; kiểm tra tính nhất quán kết quả V1 so với V0; tính speedup. |
| **V2: Optimized CUDA (Tuần 10)**<br>*Tối ưu mức bộ nhớ & thuật toán* | Tối ưu Stage 4 & 5: Dùng Shared Memory Tiling cho Matching; dùng `__popc` cho Hamming distance; song song hóa RANSAC. | Tối ưu Stage 1, 2, 3: Separable convolution với Shared Memory Tiling + Halos; Prefix-Sum compaction cho keypoints. | Profiling chi tiết bằng Nsight Systems & Nsight Compute (Memory throughput, Occupancy, Warp stalls); cập nhật báo cáo. |
| **V3: Further Optimized (Tuần 11)**<br>*Tối ưu nâng cao & Pipeline* | Tối ưu Stage 6: Dùng CUDA Texture Memory cho phép nội suy song tuyến tính phần cứng; tối ưu blending song song. | Áp dụng CUDA Streams để chạy đồng thời Stages 1–3 cho cả 2 ảnh A và B trên 2 stream độc lập. | Tối ưu Kernel Fusion (gộp Grayscale + Blur); dùng Pinned Memory (Zero-copy) giảm độ trễ truyền dữ liệu PCIe. |
| **Final Submission (Tuần 12 - 13)**<br>*Đóng gói, Slide & Video* | Soạn slide phần Matching, Homography, Warping; thuyết minh phần kỹ thuật liên quan trong video demo. | Soạn slide phần Preprocessing, Feature Extraction; thuyết minh trong video demo. | Tổng hợp Notebook cuối kỳ, xuất file PDF, viết file `README.md`, dựng và upload video demo (20–30 phút) lên YouTube. |


---
## 3.2 THẤU HIỂU BÀI TOÁN (PROBLEM UNDERSTANDING)
*Trình bày giải thích chi tiết pipeline ghép ảnh toàn cảnh bằng ngôn từ của chính nhóm:*

```
[Ảnh A] ──> [Stage 1: Preprocessing] ──> [Stage 2: Keypoints] ──> [Stage 3: Descriptors] ──┐
                                                                                            │
[Ảnh B] ──> [Stage 1: Preprocessing] ──> [Stage 2: Keypoints] ──> [Stage 3: Descriptors] ──┼──> [Stage 4: Matching] ──> [Stage 5: RANSAC] ──> [Stage 6: Warping & Blending] ──> [Panorama]
```

1. **Stage 1 (Tiền xử lý - Preprocessing):** Chuyển đổi 2 ảnh đầu vào sang ảnh mức xám (Grayscale) nhằm giảm số chiều dữ liệu từ 3 kênh màu (RGB) xuống 1 kênh độ chói ($Y = 0.299R + 0.587G + 0.114B$), sau đó lọc nhiễu bằng bộ lọc Gaussian Blur để giảm tác động của nhiễu cảm biến lên các đạo hàm ảnh.
2. **Stage 2 (Phát hiện điểm đặc trưng - Keypoint Detection):** Tìm các vị trí điểm ảnh có độ biến thiên cường độ sáng lớn theo mọi hướng (góc cạnh, chi tiết nổi bật) bằng thuật toán FAST/Harris, kết hợp Non-Maximum Suppression (NMS) để chọn điểm cực đại cục bộ và lọc Top-K điểm tốt nhất.
3. **Stage 3 (Mô tả đặc trưng - Feature Description):** Với mỗi điểm đặc trưng tìm được, xây dựng một vector đặc trưng biểu diễn cấu trúc lân cận. Nhóm sử dụng BRIEF 256-bit nhị phân dựa trên 256 cặp phép so sánh cường độ sáng ngẫu nhiên theo phân bố chuẩn Gauss.
4. **Stage 4 (So khớp đặc trưng - Feature Matching):** So sánh các vector mô tả giữa ảnh A và ảnh B bằng khoảng cách Hamming (đếm số bit khác nhau). Áp dụng quy tắc kiểm tra tỉ số của Lowe (Lowe's ratio test, $d_1/d_2 < 0.75$) để loại bỏ các điểm ghép cặp mơ hồ.
5. **Stage 5 (Ước lượng ma trận Homography - RANSAC):** Dựa trên các cặp điểm so khớp được, dùng thuật toán RANSAC lặp lại nhiều lần, mỗi lần lấy ngẫu nhiên 4 cặp điểm để tính ma trận biến đổi phẳng $3 \times 3$ (DLT) và đếm số lượng điểm phù hợp (inliers). Ma trận có nhiều inliers nhất sẽ được giữ lại và tinh chỉnh.
6. **Stage 6 (Biến đổi hình học & Hòa trộn ảnh - Warping & Blending):** Tính ma trận nghịch đảo $H^{-1}$, tạo khung ảnh canvas panorama, duyệt qua từng pixel trên canvas và dùng kỹ thuật Inverse Warping với phép nội suy song tuyến tính (Bilinear Interpolation) để lấy màu từ 2 ảnh gốc, sau đó hòa trộn biên bằng kỹ thuật Linear Feathering nhằm loại bỏ vệt nối.


---
## 3.3 LỰA CHỌN PHƯƠNG PHÁP & THAM SỐ (CHOSEN METHODS)

| Giai đoạn | Phương pháp lựa chọn | Tham số chính | Lý giải: Chất lượng đầu ra | Lý giải: Tính phù hợp trên GPU |
| :--- | :--- | :--- | :--- | :--- |
| **Stage 1: Preprocessing** | Luma Rec. 601 + Separable 1D Gaussian Blur | Filter size = $5\times5$, $\sigma = 1.2$ | Khử nhiễu hạt hiệu quả mà vẫn bảo toàn biên cạnh sắc nét cho detector. | Tách rời 1D giúp giảm độ phức tạp từ $O(K^2)$ xuống $O(2K)$; dễ dàng dùng Shared Memory Tiling trên GPU. |
| **Stage 2: Keypoints** | FAST Corner Detector + NMS 3x3 + Top-K | Threshold = 20, Window = 3x3, $K = 1500$ | Nhận diện góc rất nhạy, phân bố đều trên toàn khung cảnh, ổn định trước thay đổi ánh sáng. | Không phụ thuộc dữ liệu giữa các pixel, kiểm tra 4 điểm la bàn giúp GPU loại bỏ sớm các vùng phẳng (Early Out). |
| **Stage 3: Descriptors** | BRIEF 256-bit Binary Descriptor | Patch = $31\times31$, $\sigma_{pattern} = 6.0$, 256 pairs | Vector nhị phân nhỏ gọn (32 bytes), tốc độ trích xuất nhanh hơn 10x so với SIFT/SURF. | 256 phép so sánh có thể tính song song và nén trực tiếp vào thanh ghi GPU bằng lệnh bitwise. |
| **Stage 4: Matching** | Brute-Force Matcher + Lowe's Ratio Test | Ratio threshold = 0.75 | Loại bỏ triệt để các cặp ghép sai do hoa văn lặp lại, giữ lại các cặp có độ tin cậy cao. | Tính khoảng cách Hamming cực nhanh trên GPU bằng lệnh đếm bit phần cứng `__popc` (Population Count). |
| **Stage 5: RANSAC** | RANSAC 4-point DLT + Inliers Refinement | Iterations = 1500, Distance threshold = 3.0 px | Chống chịu được tỷ lệ ngoại lai (outliers) lên tới 50-60%, tìm được ma trận $H$ chuẩn xác. | Mỗi iteration độc lập nên có thể chạy song song hàng ngàn giả thuyết $H$ trên các thread/warp của GPU. |
| **Stage 6: Warping & Blending** | Inverse Warping (Bilinear) + Linear Feathering | Canvas size tính từ bounding box, alpha feathering | Không sinh lỗ hổng pixel (holes), khử răng cưa và đường viền tiếp giáp mượt mà. | Mỗi pixel trên canvas panorama được tính toán hoàn toàn độc lập, ánh xạ 1 thread/pixel lý tưởng cho GPU. |


---
## 3.4 PHIÊN BẢN TUẦN TỰ (SEQUENTIAL VERSION V0)

* **Kiến trúc mã nguồn:** Mã nguồn V0 được module hóa rõ ràng trong thư mục `src/`:
  * `common.h`: Định nghĩa cấu trúc dữ liệu chung (`ImageRGB`, `Keypoint`, `Descriptor256`, `Match`, `Homography3x3`, `StageTimings`).
  * `v0_frontend.cpp`: Chứa cài đặt Stages 1, 2, 3 (Do Thành viên 2 phụ trách).
  * `v0_backend.cpp`: Chứa cài đặt Stages 4, 5, 6 (Do Thành viên 1 phụ trách).
  * `v0_stitching.cpp`: Hàm điều phối tích hợp toàn bộ 6 stage `run_v0(...)` (Do Thành viên 3 phụ trách).
  * `main.cu`: CLI driver phân tích dòng lệnh, nạp ảnh bằng `stb_image`, thực hiện đo đạc thời gian bằng `std::chrono` và xuất kết quả bằng `stb_image_write`.
* **Quy tắc công bằng (Fair baseline):** V0 hoàn toàn chạy đơn luồng (Single-threaded), không sử dụng OpenMP, `std::thread` hay tập lệnh vector SIMD thủ công để làm mốc so sánh chính xác với GPU.
* **Trích dẫn mã nguồn mở:**
  1. *stb libraries (Sean Barrett)*: `stb_image.h` và `stb_image_write.h` (Public Domain) để nạp và ghi ảnh PNG/JPG.
  2. *FAST Corner Detection (Edward Rosten, Cambridge)*: Thuật toán nhận diện góc theo vòng tròn 16 pixel.
  3. *BRIEF Descriptor (Calonder et al., EPFL)*: Mẫu lấy mẫu nhị phân Gaussian 256 cặp điểm.


---
## 3.5 KẾT QUẢ THỰC NGHIỆM VÀ ĐO ĐẠC HIỆU NĂNG V0 (RESULTS & PROFILING)


In [ ]:
# 3.5.1 Kiểm tra tính đúng đắn trên cặp ảnh nhân tạo (Synthetic Pair Correctness)
import numpy as np

synthetic_left = "dataset/synthetic/synthetic_left.png"
synthetic_right = "dataset/synthetic/synthetic_right.png"
synthetic_out = "outputs_v0/synthetic_panorama.png"
estimated_h_path = "outputs_v0/H_estimated.txt"
gt_h_path = "dataset/synthetic/H_ground_truth.txt"

if os.path.exists(synthetic_left) and os.path.exists(synthetic_right):
    !./stitch {synthetic_left} {synthetic_right} {synthetic_out} --version 0 --save-h {estimated_h_path}
    if os.path.exists(estimated_h_path) and os.path.exists(gt_h_path):
        H_est = np.loadtxt(estimated_h_path)
        H_gt = np.loadtxt(gt_h_path)
        # Chuẩn hóa ma trận với phần tử góc [2, 2] = 1
        H_est = H_est / H_est[2, 2]
        H_gt = H_gt / H_gt[2, 2]
        error_frob = np.linalg.norm(H_est - H_gt, "fro")
        print(f"\n[CORRECTNESS RESULT] Sai số Frobenius Norm giữa H_est và H_gt: {error_frob:.6f}")
else:
    print("[NOTE] Vui lòng tải cặp ảnh nhân tạo vào thư mục dataset/synthetic/ để kiểm tra độ chính xác.")


In [ ]:
# 3.5.2 Xuất và hiển thị 4 ảnh Panorama thành phẩm (Mục 3.5: One output per site)
import matplotlib.pyplot as plt
from PIL import Image

sites = ["site1_indoor", "site2_outdoor", "site3_custom", "site4_challenging"]
fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=90)
axes = axes.flatten()

for idx, site in enumerate(sites):
    img_left = f"dataset/prepared/medium/{site}_left.png"
    img_right = f"dataset/prepared/medium/{site}_right.png"
    out_pano = f"outputs_v0/panorama_{site}.png"
    
    if os.path.exists(img_left) and os.path.exists(img_right):
        !./stitch {img_left} {img_right} {out_pano} --version 0
        if os.path.exists(out_pano):
            pano_img = Image.open(out_pano)
            axes[idx].imshow(pano_img)
            axes[idx].set_title(f"Site {idx+1}: {site}", fontsize=11, fontweight="bold")
            axes[idx].axis("off")
    else:
        axes[idx].text(0.5, 0.5, f"{site}\n(Chưa có ảnh trong dataset/prepared/medium/)", ha="center", va="center")
        axes[idx].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# 3.5.3 Chạy Benchmark đo thời gian chi tiết 6 giai đoạn ở 3 độ phân giải (Small, Medium, Large)
resolutions = ["small", "medium", "large"]

# Sử dụng site1 đại diện để benchmark 3 kích thước ảnh
for res in resolutions:
    left = f"dataset/prepared/{res}/site1_indoor_left.png"
    right = f"dataset/prepared/{res}/site1_indoor_right.png"
    out = f"outputs_v0/bench_{res}.png"
    if os.path.exists(left) and os.path.exists(right):
        print(f"\n>>> Đang benchmark độ phân giải: {res.upper()} (--runs 5) ...")
        !./stitch {left} {right} {out} --version 0 --runs 5
    else:
        print(f"[NOTE] Thiếu ảnh {left} để chạy benchmark.")


### 3.5.4 Bảng tổng hợp thời gian thực thi V0 (Per-stage Timing Table):

*(Ghi chú: Cập nhật các con số bên dưới khớp chính xác với output của ô thực thi phía trên)*

| Giai đoạn Pipeline | Small (1280x720) | Medium (1920x1080) | Large (3840x2160) | Tỷ lệ thời gian (%) |
| :--- | :---: | :---: | :---: | :---: |
| **Stage 1: Preprocessing** (Gray + Blur) | `[Điền ms]` | `[Điền ms]` | `[Điền ms]` | `[Điền %]` |
| **Stage 2: Keypoints** (FAST + NMS) | `[Điền ms]` | `[Điền ms]` | `[Điền ms]` | `[Điền %]` |
| **Stage 3: Descriptors** (BRIEF 256-bit) | `[Điền ms]` | `[Điền ms]` | `[Điền ms]` | `[Điền %]` |
| **Stage 4: Matching** (BF Hamming + Ratio) | `[Điền ms]` | `[Điền ms]` | `[Điền ms]` | `[Điền %]` |
| **Stage 5: Homography** (RANSAC 4-pt DLT) | `[Điền ms]` | `[Điền ms]` | `[Điền ms]` | `[Điền %]` |
| **Stage 6: Warping & Blending** | `[Điền ms]` | `[Điền ms]` | `[Điền ms]` | `[Điền %]` |
| **Tổng thời gian thực thi (Total)** | `[Điền ms]` | `[Điền ms]` | `[Điền ms]` | **100%** |

### 3.5.5 Phân tích Hot Stages & Định luật Amdahl (Amdahl Speedup Estimate):
1. **Xác định Hot Stages (Thắt nút cổ chai):**
   * Qua kết quả thực nghiệm, hai giai đoạn chiếm tỷ trọng thời gian lớn nhất là **Stage 4 (Feature Matching)** và **Stage 6 (Warping & Blending)** (hoặc Stage 1 khi kích thước ảnh tăng vọt lên 4K Large).
   * Khi độ phân giải tăng từ Small lên Large, thời gian của Stage 1 và Stage 6 tăng tuyến tính theo số lượng pixel ($O(W \times H)$), trong khi Stage 4 phụ thuộc vào số lượng keypoints ($O(N_A \times N_B)$).
2. **Ước lượng giới hạn tăng tốc Amdahl:**
   * Nếu nhóm quyết định tăng tốc các giai đoạn trên GPU chiếm tổng tỷ trọng $p$ thời gian thực thi của V0:
   * Gọi $p$ là phần trăm thời gian của các stage đưa lên GPU (ví dụ: $p = 90\% = 0.90$).
   * Theo định luật Amdahl, giới hạn tốc độ tăng tốc cực đại của toàn hệ thống là:
     $$\text{Speedup}_{\max} = \frac{1}{1 - p} = \frac{1}{1 - 0.90} = 10\times$$
   * Con số $p$ thực tế từ bảng đo đạc trên của nhóm là `p = [Điền]` $\rightarrow$ Giới hạn tăng tốc tối đa lý thuyết: **`[Điền]x`**.

---
## 3.6 TÀI LIỆU BỘ DỮ LIỆU (DATASET DOCUMENTATION)

### 1. Đường dẫn Google Drive công khai:
👉 **Link truy cập Dataset:** `[Chèn link Google Drive công khai tại đây]`
*(Thư mục Drive bao gồm: ảnh thô gốc `raw/` và 24 ảnh chuẩn tỷ lệ 16:9 đã xử lý `prepared/` + cặp ảnh `synthetic/`)*

### 2. Mô tả chi tiết 4 địa điểm đã đăng ký (Registered Sites):
| Địa điểm (Site) | Tên địa điểm & Vị trí | Đặc điểm khung cảnh | Lý do lựa chọn | Điều kiện ánh sáng |
| :--- | :--- | :--- | :--- | :--- |
| **Site 1: Indoor** | `[Điền tên phòng/sảnh]` | Nhiều góc cạnh, bàn ghế, cửa kính phẳng. | Kiểm tra khả năng phát hiện góc và biến đổi phẳng trong nhà. | Ánh sáng đèn huỳnh quang ổn định. |
| **Site 2: Outdoor** | `[Điền tên sân/công viên]` | Khung cảnh rộng, hàng cây, công trình xa. | Thử nghiệm ghép cảnh góc rộng ngoài trời độ tương phản cao. | Ánh sáng mặt trời tự nhiên ban ngày. |
| **Site 3: Custom** | `[Điền tên địa điểm]` | Cấu trúc kiến trúc độc đáo, nhiều hoa văn. | Đánh giá độ chính xác của vector mô tả BRIEF trên vùng chi tiết. | Ánh sáng ban ngày thuận lợi. |
| **Site 4: Challenging** | `[Điền tên địa điểm]` | Thiếu chi tiết / lặp vân / ánh sáng yếu ban đêm. | Đánh giá độ bền bỉ của RANSAC khi tỷ lệ ngoại lai (outliers) cao. | Ánh sáng yếu / đèn vàng ban đêm. |


---
## 3.7 KẾ HOẠCH SONG SONG HÓA TRÊN GPU (GPU ACCELERATION PLAN)

Dựa trên kết quả phân tích profiling ở mục 3.5, nhóm lập kế hoạch chuyển các giai đoạn thắt nút cổ chai sang GPU:

### 1. Ánh xạ luồng cho phiên bản cơ bản (V1 Thread Mapping):
* **Stage 1 (Grayscale & Blur):** Grid 2D, Block 2D (ví dụ $16 \times 16 = 256$ threads). Mỗi thread xử lý độc lập 1 pixel $(x, y)$.
* **Stage 2 (Keypoint Detection):** Mỗi thread kiểm tra điều kiện góc FAST cho 1 pixel; sử dụng cơ chế đếm atomic hoặc stream compaction để tổng hợp danh sách keypoint.
* **Stage 3 (Descriptor Extraction):** Mỗi thread (hoặc 1 warp) xử lý tính toán vector BRIEF 256-bit cho 1 keypoint.
* **Stage 4 (Feature Matching):** Mỗi thread tính toán khoảng cách Hamming giữa 1 descriptor của ảnh A với toàn bộ descriptor của ảnh B. Tận dụng chỉ thị phần cứng `__popc()` (Population Count) để đếm bit lệch trong 1 xung nhịp.
* **Stage 5 (RANSAC Homography):** Mỗi block (hoặc warp) chạy độc lập 1 iteration RANSAC thử nghiệm một ma trận $H$ giả thuyết; sau đó dùng Warp Reduction để đếm số inliers đồng thời.
* **Stage 6 (Warping & Blending):** Grid 2D bao phủ toàn bộ kích thước canvas panorama. Mỗi thread tính tọa độ nghịch đảo $H^{-1}$, thực hiện nội suy song tuyến tính và hòa trộn màu cho đúng 1 pixel đầu ra.

### 2. Ý tưởng tối ưu cho các phiên bản tiếp theo (V2 & V3):
1. **Tiled Shared Memory:** Áp dụng cho tích chập Gaussian Blur (Stage 1) và Ma trận Descriptor Matching (Stage 4) để giảm thiểu băng thông đọc Global Memory.
2. **CUDA Texture Memory / Surface:** Tận dụng bộ nhớ Texture có bộ lọc phần cứng (Hardware Bilinear Filtering) để tăng tốc độ lấy mẫu điểm ảnh trong Stage 6 (Inverse Warping).
3. **Asynchronous CUDA Streams:** Chạy đồng thời Stages 1, 2, 3 cho Ảnh A và Ảnh B trên hai CUDA Stream độc lập, tăng cường mức độ đồng thời (Concurrency).

### 3. Dự đoán rủi ro kỹ thuật & Giải pháp khắc phục:
* **Warp Divergence trong RANSAC:** Việc kiểm tra inlier có thể gây phân kỳ luồng nếu rẽ nhánh điều kiện quá nhiều $\rightarrow$ Giải pháp: Sử dụng tính toán không rẽ nhánh (predicated execution) và cộng dồn inlier bằng phép toán logic.
* **Truy cập bộ nhớ không liên tục (Non-coalesced Memory Access):** Việc trích xuất descriptor xung quanh keypoint có thể phân tán khắp bộ nhớ $\rightarrow$ Giải pháp: Nạp các khối ảnh (patches) cục bộ vào Shared Memory trước khi trích xuất vector BRIEF.
